In [5]:
import pandas as pd
import numpy as np

# Phase 1: Load files using the exact filenames matching your workspace directory
try:
    df_interactions = pd.read_csv('social_media_interactions_contaminated.csv')
    df_transactions = pd.read_csv('customer_transactions_contaminated.csv')
    df_customers = pd.read_csv('customer_demographics_contaminated.csv')
    print("SUCCESS: All contaminated datasets parsed and loaded securely into working memory arrays!")
except FileNotFoundError as err:
    print(f"ERROR: {err}\nPlease double check the exact spelling of your filenames.")

# Run initial row validation duplicates inspections checks loop tracking framework metrics
if 'df_customers' in locals():
    print(f"\nDuplicate Row Counts Detected Across Framework Tables:")
    print(f"* Interactions table exact duplicate record frequency count: {df_interactions.duplicated().sum()}")
    print(f"* Transactions table exact duplicate record frequency count: {df_transactions.duplicated().sum()}")
    print(f"* Customers table exact duplicate record frequency count: {df_customers.duplicated().sum()}")
    
    print("\n--- Initial Inspection: Customer Demographics Structural Columns Meta Summary ---")
    df_customers.info()


SUCCESS: All contaminated datasets parsed and loaded securely into working memory arrays!

Duplicate Row Counts Detected Across Framework Tables:
* Interactions table exact duplicate record frequency count: 180
* Transactions table exact duplicate record frequency count: 185
* Customers table exact duplicate record frequency count: 177

--- Initial Inspection: Customer Demographics Structural Columns Meta Summary ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   CustomerID   3200 non-null   object
 1   Age          2909 non-null   object
 2   Gender       3200 non-null   object
 3   Location     3200 non-null   object
 4   IncomeLevel  2897 non-null   object
 5   SignupDate   3200 non-null   object
dtypes: object(6)
memory usage: 150.1+ KB


In [6]:
# 1. Drop the exact duplicate rows we discovered in Phase 1
df_interactions.drop_duplicates(inplace=True)
df_transactions.drop_duplicates(inplace=True)
df_customers.drop_duplicates(inplace=True)

# 2. Fix Customers 'Age': Convert text to numbers, treat errors (like text strings) as NaN
df_customers['Age'] = pd.to_numeric(df_customers['Age'], errors='coerce')

# 3. Handle impossible customer ages by filtering out values below 1 or above 100
df_customers.loc[(df_customers['Age'] < 1) | (df_customers['Age'] > 100), 'Age'] = np.nan

# 4. Impute missing customer ages using the dataset median age calculation
df_customers['Age'] = df_customers['Age'].fillna(df_customers['Age'].median())

# 5. Handle missing string classifications by using a placeholder text string
df_customers['IncomeLevel'] = df_customers['IncomeLevel'].fillna('Unknown')

# 6. Standardize date columns across all tables into YYYY-MM-DD timestamp structures
df_customers['SignupDate'] = pd.to_datetime(df_customers['SignupDate'], errors='coerce')
df_interactions['InteractionDate'] = pd.to_datetime(df_interactions['InteractionDate'], errors='coerce')
df_transactions['TransactionDate'] = pd.to_datetime(df_transactions['TransactionDate'], errors='coerce')

# Print out verification check to confirm updates worked cleanly
print("--- Cleaned Customers Structural Meta Summary ---")
df_customers.info()


--- Cleaned Customers Structural Meta Summary ---
<class 'pandas.core.frame.DataFrame'>
Index: 3023 entries, 0 to 3195
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   CustomerID   3023 non-null   object        
 1   Age          3023 non-null   float64       
 2   Gender       3023 non-null   object        
 3   Location     3023 non-null   object        
 4   IncomeLevel  3023 non-null   object        
 5   SignupDate   2925 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(4)
memory usage: 165.3+ KB


In [7]:
# 1. Clean the transaction Amount column: convert text placeholder values like 'Free' to 0
if 'Amount' in df_transactions.columns:
    df_transactions['Amount'] = df_transactions['Amount'].replace('Free', 0)
    df_transactions['Amount'] = pd.to_numeric(df_transactions['Amount'], errors='coerce')
    
    # Take absolute value to fix negative transaction errors (e.g., -100 becomes 100)
    df_transactions['Amount'] = df_transactions['Amount'].abs()
    
    # Fill remaining missing transactions with the median value
    df_transactions['Amount'] = df_transactions['Amount'].fillna(df_transactions['Amount'].median())

# 2. Fill missing text category fields with clean standard placeholders
df_transactions['ProductCategory'] = df_transactions['ProductCategory'].fillna('Uncategorized')
df_interactions['Platform'] = df_interactions['Platform'].fillna('Unknown')
df_interactions['Sentiment'] = df_interactions['Sentiment'].fillna('Neutral')

# 3. Export clean production-ready copies exactly as required by your submission guidelines
df_interactions.to_csv('finmark_cleaned_interactions.csv', index=False)
df_transactions.to_csv('finmark_cleaned_transactions.csv', index=False)
df_customers.to_csv('finmark_cleaned_customers.csv', index=False)

print("🎉 SUCCESS: Clean datasets exported successfully to your project directory folder!")


🎉 SUCCESS: Clean datasets exported successfully to your project directory folder!
